# 1. Define and explore SysML v2 models

`sysml2` parses the OMG SysML v2 **textual notation** into a typed Python
object model. The parsers are generated with ANTLR from the SysML v2 and
KerML grammars.

In [ ]:
import sysml2

model = sysml2.loads('''
package Vehicles {
    doc /* A small demonstration model. */

    part def Wheel { attribute diameter : Real = 0.66; }

    part def Vehicle {
        attribute mass : Real = 1200.0;
        attribute maxMass : Real = 2000.0;
        part wheels : Wheel[4];
        assert constraint massLimit { mass <= maxMass }
    }
}
''')
model

## The model is a tree of dataclasses

Every element has a `kind`, a `qualified_name`, and typed fields
(`typing.Literal` vocabularies for kinds, directions, visibilities, ...).

In [ ]:
for element in model.iter_tree():
    kind = getattr(element, "kind", type(element).__name__)
    print(f"{'  ' * len((element.qualified_name or '').split('::'))}"
          f"{kind:12s} {element.qualified_name or ''}")

In [ ]:
vehicle = model.find("Vehicles::Vehicle")
mass = model.find("Vehicles::Vehicle::mass")
print("kind:      ", mass.kind)
print("types:     ", mass.types)
print("value expr:", mass.value.expr.to_text())
print("supers of Vehicle:", vehicle.supers)
print("doc:", model.find("Vehicles").doc)

## Models can be built programmatically

The same dataclasses are the authoring API -- no text required.
Expressions come from `sysml2.parse_expression`.

In [ ]:
from sysml2 import model as M

pkg = M.Package(name="Generated")
sensor = M.Definition(kind="part", name="Sensor")
sensor.add(
    M.Usage(kind="attribute", name="rate", types=["Real"],
            value=M.FeatureValue(sysml2.parse_expression("100.0 * 2"))),
    M.Usage(kind="attribute", name="enabled", types=["Boolean"],
            value=M.FeatureValue(sysml2.parse_expression("true"))),
)
pkg.add(sensor)

generated = M.Model()
generated.add(pkg)
print(sysml2.to_sysml(generated))

## Multi-file workspaces

`sysml2.load()` accepts a single `.sysml` file, a `.json` export, or a
**directory**: all files merge under one root namespace, so cross-file
imports resolve. Built models are cached as JSON (content-addressed, keyed
on source + code fingerprints), which makes warm loads ~1000x faster than
the ANTLR Python runtime's cold parse.

In [ ]:
import tempfile
from pathlib import Path

workspace = Path(tempfile.mkdtemp())
(workspace / "units.sysml").write_text(
    "package Units { attribute gravity : Real = 9.81; }")
(workspace / "app.sysml").write_text('''
package App {
    private import Units::*;
    calc def Weight { in m : Real; return : Real = m * gravity; }
}
''')

merged = sysml2.load(workspace)          # directory -> merged model
interp = sysml2.Interpreter(merged)
interp.call("App::Weight", m=10.0)       # cross-file resolution works